# LAYA Colab Runs: loop test, sufficiency race, W1 retrain

This notebook runs three registered experiments for the LAYA study, in
order. Each run has its own board-passed PRD in the study record
(`swarm_tests/loop-test/PRD.md`, `swarm_tests/sufficiency-race/PRD.md`,
`swarm_tests/w1-retrain/PRD.md`). This notebook implements those PRDs
exactly. It does not invent measurements: every results file contains
exactly the registered legs of its PRD, and nothing else is scored.

Run order, which is a dependency order, not a preference:

1. **Run 1, the loop test (H-H).** One second judging pass whose context
   carries the pass-1 probability state, on the frozen 300-item
   NewsCLIPpings probe.
2. **Run 2, the sufficiency race (N5).** Three abstention triggers raced
   on the loop's frozen probes. It consumes Run 1's outputs and will
   refuse to start without them.
3. **Run 3, the W1 retrain.** The exact recorded 245-step recipe that
   rebuilds the lost MMFakeBench Phase 3 checkpoint and unblocks the
   loss-masking audit.

Rules that hold in every cell:

- The MMFakeBench test set is CLOSED. It was scored exactly once. No
  cell in this notebook reads it, scores it, or touches it. Runs read
  validation, tune, and frozen probe artifacts only.
- Cost is $0. This notebook targets a free Colab GPU.
- Progress is checkpointed to Google Drive at least every 30 minutes
  of wall time, and every checkpoint save is verified (size and
  sha256 read back and compared) before the run continues. A failed
  verify raises an error and retries; it never silently continues.
  This is the Amendment 14 lesson: an unverified checkpoint on a
  dead session is a lost checkpoint.
- If a session dies, reopen this notebook, re-run the Setup cells and
  the run's RESUME cell, then re-run that run's cells. Completed work
  is skipped by item, exactly as the study's judging scripts do.

## Setup

What this section does: mounts Google Drive, installs the study's
pinned `laya` package, loads the study's own judging and scoring code
from the inputs bundle, and checks every input the three runs need.
The check is a gate: if anything is missing, setup stops and names
what is missing. Nothing runs on partial inputs.

What it reads: the Drive folder `LAYA_Colab/inputs/`, which holds a
`code/` folder (the study's scripts, copied unchanged), the frozen
probe items and their byte-exact images, the checkpoints, the stored
pass-1 prediction files, the scale carve's tune items, the trainer
records for feature extraction, and the MMFakeBench train and tune
files for Run 3.

What it writes: nothing except a local working copy of the code and a
printed inputs report.

In [ ]:
# ---- Setup 1: mount Drive, define paths, check the GPU ----
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, hashlib
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/LAYA_Colab')
INPUTS     = DRIVE_ROOT / 'inputs'
CKPT_ROOT  = DRIVE_ROOT / 'LAYA_Colab_Checkpoints'   # verified checkpoints live here
RESULTS    = DRIVE_ROOT / 'results'
WORK       = Path('/content/laya_work')              # local scratch on the VM
for p in (CKPT_ROOT, RESULTS, WORK):
    p.mkdir(parents=True, exist_ok=True)

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU in this session. Judging will be very slow. '
          'Use Runtime > Change runtime type > GPU, then re-run setup.')

In [ ]:
# ---- Setup 2: pinned installs ----
# The study's dev laya package, at the exact commit the study verified
# (laya_trainer.py refuses any other build). PyPI 'laya' has no VLM code.
%pip install -q "git+https://github.com/r33drichards/laya-vision@9e1e2419"
%pip install -q "scikit-learn>=1.3"

import laya
from laya import vlm_train
for name in ('train', 'jsonl_example', 'collect_logits',
             'fit_temperatures_from', 'metrics_from'):
    assert hasattr(vlm_train, name), f'laya.vlm_train has no {name}: wrong package build'
assert hasattr(laya, 'load_vlm')
print('laya dev package verified: load_vlm + vlm_train surface OK')

In [ ]:
# ---- Setup 3: load the study code bundle and gate on every input ----
import shutil
CODE = INPUTS / 'code'
for f in ('judge_pilot.py', 'audit_scorer.py', 'visualnews_stream.py',
          'laya_trainer.py', 'train_mmf.py'):
    src = CODE / f
    assert src.exists(), f'MISSING study code file: {src}'
    shutil.copy(src, WORK / f)
sys.path.insert(0, str(WORK))

REQUIRED = {
    'probe items (frozen 300)':            INPUTS / 'probe_items.jsonl',
    'stored pass-1 preds (scale probe)':   INPUTS / 'preds_scale_probe_300.jsonl',
    'stored pass-1 preds (Stage A)':       INPUTS / 'preds_stageA_300.jsonl',
    'scale probe checkpoint dir':          INPUTS / 'ckpts' / 'scale_probe',
    'Stage B checkpoint dir':              INPUTS / 'ckpts' / 'stageb',
    'Stage A winner checkpoint dir':       INPUTS / 'ckpts' / 'stagea_winner',
    'scale tune items (200)':              INPUTS / 'scale_tune_items.jsonl',
    'probe trainer records (Run 2)':       INPUTS / 'probe_records.jsonl',
    'scale tune trainer records (Run 2)':  INPUTS / 'scale_tune_records.jsonl',
    'W1 base checkpoint dir':              INPUTS / 'ckpts' / 'laya-vision-base',
    'MMF train file (980 records)':        INPUTS / 'mmf_train_base.jsonl',
    'MMF tune file':                       INPUTS / 'mmf_tune_base.jsonl',
}
missing = [k for k, p in REQUIRED.items() if not p.exists()]
for k, p in REQUIRED.items():
    print(('OK      ' if p.exists() else 'MISSING ') + k)
probe_images = INPUTS / 'probe_images'
n_img = sum(1 for _ in probe_images.rglob('*') if _.is_file()) if probe_images.exists() else 0
print(f'probe images present: {n_img} files under {probe_images}')
if n_img < 300:
    missing.append(f'probe images (found {n_img}, need the 300 byte-exact members)')
if missing:
    raise RuntimeError('INPUTS GATE FAILED, missing: ' + '; '.join(missing))
print('INPUTS GATE PASSED')

## The checkpoint keeper (used by every run)

What this section does: defines the one checkpoint mechanism all three
runs share. A checkpoint is a copy on Drive of the run's progress (for
judging runs, the predictions file plus a manifest; for Run 3, the
finished model folder), saved at least every 30 minutes of wall time
and at the end of every run.

Every save is verified immediately after writing: the copy's size and
sha256 are read back from Drive and compared with the source. A failed
verify retries the save, and raises after three failures. A save is
never reported as done until it has been read back and matched.

What it reads and writes: local progress files in `/content/laya_work`,
and verified copies under `LAYA_Colab/LAYA_Colab_Checkpoints/<run-slug>/`.

In [ ]:
# ---- Checkpoint keeper: 30-minute saves, verify-on-save, resume ----
import hashlib, json, shutil, time
from pathlib import Path

CKPT_INTERVAL_S = 30 * 60   # Rafiul's rule, 2026-11-27: a checkpoint every 30 minutes

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def safetensors_header_check(path):
    '''The loss-masking audit step-0 check: file size must cover
    8 + header length + the header's declared data end.'''
    import struct
    path = Path(path)
    size = path.stat().st_size
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]
        header = json.loads(f.read(n))
    data_end = 0
    n_tensors = 0
    for name, meta in header.items():
        if name == '__metadata__':
            continue
        n_tensors += 1
        data_end = max(data_end, meta['data_offsets'][1])
    ok = size >= 8 + n + data_end
    return {'ok': ok, 'bytes': size, 'tensors': n_tensors}

class Keeper:
    '''Verified checkpointing to Drive for one run.'''
    def __init__(self, run_slug):
        self.dir = CKPT_ROOT / run_slug
        self.dir.mkdir(parents=True, exist_ok=True)
        self.last_save = time.monotonic()

    def save_verified(self, local_path, name=None):
        '''Copy local_path into this run's Drive checkpoint folder and
        prove the copy: size and sha256 read back and compared.
        Retries 3 times, then raises. Never silently continues.'''
        local_path = Path(local_path)
        name = name or local_path.name
        want_size = local_path.stat().st_size
        want_hash = sha256_of(local_path)
        dest = self.dir / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(1, 4):
            tmp = self.dir / (name + f'.tmp{attempt}')
            shutil.copy(local_path, tmp)
            if tmp.stat().st_size == want_size and sha256_of(tmp) == want_hash:
                tmp.replace(dest)
                self.last_save = time.monotonic()
                print(f'[keeper] verified save: {name} '
                      f'({want_size} bytes, sha256 {want_hash[:12]}...)')
                return dest
            print(f'[keeper] VERIFY FAILED for {name} (attempt {attempt}); retrying')
            tmp.unlink(missing_ok=True)
        raise IOError(f'checkpoint save could not be verified after 3 attempts: {name}')

    def save_manifest(self, manifest: dict):
        p = WORK / 'manifest.json'
        p.write_text(json.dumps(manifest, indent=1))
        return self.save_verified(p, 'manifest.json')

    def maybe_save(self, local_paths, manifest=None):
        '''Call inside long loops: saves when 30 minutes have elapsed.'''
        if time.monotonic() - self.last_save >= CKPT_INTERVAL_S:
            for p in local_paths:
                if Path(p).exists():
                    self.save_verified(p)
            if manifest is not None:
                self.save_manifest(manifest)
            print('[keeper] 30-minute checkpoint complete')

    def restore(self, name, local_path):
        '''Copy the newest verified checkpoint back to local and prove it.'''
        src = self.dir / name
        if not src.exists():
            print(f'[keeper] no checkpoint named {name} on Drive; starting fresh')
            return False
        local_path = Path(local_path)
        shutil.copy(src, local_path)
        if sha256_of(local_path) != sha256_of(src):
            raise IOError(f'restored checkpoint failed verification: {name}')
        print(f'[keeper] restored verified checkpoint: {name}')
        return True

def config_hash(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:16]

print('Keeper ready. Interval:', CKPT_INTERVAL_S // 60, 'minutes.')

## Run 1: the loop test (H-H)

What this run does: exactly one second judging pass over the frozen
300-item NewsCLIPpings probe, where the second pass's context carries
the pass-1 probability state and nothing else new. PRD:
`swarm_tests/loop-test/PRD.md`. The registered legs, all four of which
must pass together on the primary read (scale probe checkpoint,
loop-all arm): uncertain count below 137; headline accuracy above
0.0467 with its paired resampling interval above the pass-1 read;
accuracy including uncertain at or above 0.5000; falsified headline
accuracy at or above 0.0811. Strong form: headline at or above 0.4300.

What it reads: the frozen probe items and images, the scale probe
checkpoint (primary), the Stage B and Stage A winner checkpoints
(secondary, descriptive only), and the stored pass-1 predictions for
the reproduction gate.

What it writes: prediction files per arm under the working folder,
checkpointed to Drive as it goes, and at the end a results JSON with
exactly the registered legs, to Drive and printed below.

The pass-2 state block is fixed verbatim in the PRD. Do not edit its
wording in this notebook: a wording change is a new registration.

In [ ]:
# ---- Run 1 shared judging helper (the study's judge_disk pattern) ----
import json, time
import torch
import laya
from PIL import Image
import visualnews_stream as vs
from judge_pilot import V1_NATIVE, V1_FAKE, VCFG, decode_answers
import audit_scorer as A

LABELS = ('pristine', 'falsified')
ITEMS = [json.loads(l) for l in open(INPUTS / 'probe_items.jsonl')]
ITEM_BY_ID = {it['id']: it for it in ITEMS}
PROBE_IMAGES = INPUTS / 'probe_images'

def load_agent(ckpt_dir):
    agent = laya.load_vlm(str(ckpt_dir), device='cuda', dtype='bfloat16',
                          max_len=1024, head_max_len=256)
    print('temperature as published:', [float(x) for x in agent.temperature])
    agent.temperature = [1.0] * len(agent.temperature)  # registered probe setting
    return agent

def judge_rows(agent, rows_spec, out_path, keeper, run_slug):
    '''rows_spec: list of (item, arm, context). Rows are flushed per item;
    (id, arm) pairs already in out_path are skipped, so a re-run resumes.
    The keeper checkpoints the predictions file at least every 30 min.'''
    out_path = Path(out_path)
    done = set()
    if out_path.exists():
        for line in open(out_path):
            r = json.loads(line)
            if r.get('status') == 'ok':
                done.add((r['id'], r['arm']))
    out = open(out_path, 'a')
    n_new = 0
    for it, arm, context in rows_spec:
        if (it['id'], arm) in done:
            continue
        p = PROBE_IMAGES / it['image_path']
        ok, reason, img = vs.verify_image_bytes(p.read_bytes())
        if not ok:
            out.write(json.dumps({'id': it['id'], 'arm': arm, 'gold': it['gold'],
                                  'run_tag': run_slug, 'status': f'skip:{reason}'}) + '\n')
            out.flush()
            continue
        t0 = time.monotonic()
        with torch.inference_mode():
            result = agent.predict({'context': context, 'image': img},
                                   {'native': V1_NATIVE, 'fake': V1_FAKE},
                                   n_permutations=VCFG['n_permutations'],
                                   batch_size=VCFG['batch_size'],
                                   prefix_cache=VCFG['prefix_cache'],
                                   temperature=VCFG['temperature'],
                                   strict=VCFG['strict'])
        dec = decode_answers(result, LABELS)
        out.write(json.dumps({'id': it['id'], 'arm': arm, 'gold': it['gold'],
                              'run_tag': run_slug, 'route': 'pair_only',
                              'status': 'ok',
                              'seconds': round(time.monotonic() - t0, 4), **dec}) + '\n')
        out.flush()
        n_new += 1
        keeper.maybe_save([out_path], manifest={
            'run': run_slug, 'rows_done': len(done) + n_new,
            'config_hash': config_hash({'items': len(ITEMS), 'arm': arm})})
    out.close()
    keeper.save_verified(out_path)
    print(f'{run_slug}: judged {n_new} new rows -> {out_path.name}')
    return out_path

def read_rows(path, arm):
    rows = {}
    for line in open(path):
        r = json.loads(line)
        if r.get('status') == 'ok' and r['arm'] == arm:
            rows[r['id']] = r
    return rows

def arm_scores(rows, ids):
    '''The audited reads (audit_scorer verdict at THR_A 0.5/0.5).'''
    s = A.acc(rows, ids, A.THR_A)
    per = {}
    for gold in LABELS:
        gids = [i for i in ids if rows[i]['gold'] == gold]
        okc = sum(1 for i in gids
                  if A.verdict(rows[i], A.THR_A)[0] == gold
                  and not A.verdict(rows[i], A.THR_A)[1])
        per[gold] = round(okc / len(gids), 4) if gids else None
    return {'headline_acc': round(s['acc_headline'], 4),
            'acc_incl_uncertain': round(s['acc_incl_uncertain'], 4),
            'n_uncertain': s['n_uncertain'], 'n': len(ids),
            'per_class_headline': per}

def paired_resampling_delta(rows_a, rows_b, ids, resamples=10000, seed=20261122):
    '''Interval for (b minus a) headline correctness, per item.
    Headline correctness: decided and correct under the audited verdict.'''
    import random
    def correct(rows, i):
        lab, unc = A.verdict(rows[i], A.THR_A)
        return 1.0 if (not unc and lab == rows[i]['gold']) else 0.0
    diffs = [correct(rows_b, i) - correct(rows_a, i) for i in ids]
    point = sum(diffs) / len(diffs)
    rng = random.Random(seed)
    n = len(diffs)
    est = []
    for _ in range(resamples):
        est.append(sum(diffs[rng.randrange(n)] for _ in range(n)) / n)
    est.sort()
    return {'delta': round(point, 4),
            'ci95': [round(est[int(0.025 * resamples)], 4),
                     round(est[int(0.975 * resamples)], 4)]}
print('Run 1 helpers ready.')

In [ ]:
# ---- Run 1, step 0+1: inputs gate and the pass-1 reproduction gate ----
RUN1 = 'loop-test'
keeper1 = Keeper(RUN1)
ckpt_scale = INPUTS / 'ckpts' / 'scale_probe'
model_file = next(ckpt_scale.rglob('model.safetensors'))
chk = safetensors_header_check(model_file)
print('scale probe checkpoint header check:', chk)
assert chk['ok'] and chk['bytes'] == 804696880 and chk['tensors'] == 416, \
    'STEP 0 GATE FAILED: scale probe checkpoint is not the recorded one'

agent_scale = load_agent(ckpt_scale)
preds1_path = WORK / 'preds_loop_pass1.jsonl'
judge_rows(agent_scale, [(it, 'P1', it['caption']) for it in ITEMS],
           preds1_path, keeper1, RUN1 + ':pass1')

# Reproduction gate against the stored pass-1 read (PRD step 1).
stored = read_rows(INPUTS / 'preds_scale_probe_300.jsonl', 'P0')
mine = read_rows(preds1_path, 'P1')
ids = [it['id'] for it in ITEMS if it['id'] in stored and it['id'] in mine]
assert len(ids) == 300, f'pass-1 reads cover {len(ids)} items, not 300'
s_mine, s_stored = arm_scores(mine, ids), arm_scores(stored, ids)
deviations = [i for i in ids
              if A.verdict(mine[i], A.THR_A) != A.verdict(stored[i], A.THR_A)]
print('pass 1, this session:', s_mine)
print('pass 1, stored read  :', s_stored)
print(f'verdict deviations vs stored: {len(deviations)} items: {deviations[:20]}')
assert abs(s_mine['headline_acc'] - 0.0467) <= 0.0033 + 1e-9, 'REPRODUCTION GATE FAILED: headline'
assert abs(s_mine['n_uncertain'] - 274) <= 2, 'REPRODUCTION GATE FAILED: uncertain count'
print('REPRODUCTION GATE PASSED: pass 1 reproduces the record inside tolerance')

In [ ]:
# ---- Run 1, steps 2+3: pass 2, loop-all arm and disagreement-gated arm ----
# The state block below is fixed verbatim in the PRD. Do not edit it here.
STATE_TEMPLATE = ('FIRST PASS RECORD. Choice question probabilities: pristine '
                  '{p_pristine:.4f}, falsified {p_falsified:.4f}. Screen question '
                  'probability that the item is falsified: {p_screen:.4f}. At the '
                  'registered thresholds the two questions {agree_word}.')

def state_block(row):
    probs = row['probabilities']
    lab, unc = A.verdict(row, A.THR_A)
    return STATE_TEMPLATE.format(
        p_pristine=probs['pristine'], p_falsified=probs['falsified'],
        p_screen=row['fake_probability'],
        agree_word='DISAGREE' if unc else 'AGREE')

pass1 = read_rows(preds1_path, 'P1')
preds2_path = WORK / 'preds_loop_pass2.jsonl'
spec_all = [(it, 'P2ALL', it['caption'] + '\n\n' + state_block(pass1[it['id']]))
            for it in ITEMS]
judge_rows(agent_scale, spec_all, preds2_path, keeper1, RUN1 + ':pass2-all')

uncertain_ids = {i for i in ids if A.verdict(pass1[i], A.THR_A)[1]}
spec_gate = [(ITEM_BY_ID[i], 'P2GATE',
              ITEM_BY_ID[i]['caption'] + '\n\n' + state_block(pass1[i]))
             for i in sorted(uncertain_ids)]
judge_rows(agent_scale, spec_gate, preds2_path, keeper1, RUN1 + ':pass2-gated')
print(f'pass 2 done: loop-all on 300, gated on {len(spec_gate)} uncertain items')

In [ ]:
# ---- Run 1, step 4: secondary reads (Stage B, Stage A winner), descriptive ----
secondary = {}
for tag, ckpt_dir in (('stageb', INPUTS / 'ckpts' / 'stageb'),
                      ('stagea', INPUTS / 'ckpts' / 'stagea_winner')):
    agent_s = load_agent(ckpt_dir)
    p1 = WORK / f'preds_loop_{tag}_pass1.jsonl'
    judge_rows(agent_s, [(it, 'P1', it['caption']) for it in ITEMS],
               p1, keeper1, f'{RUN1}:{tag}-pass1')
    r1 = read_rows(p1, 'P1')
    spec = [(it, 'P2ALL', it['caption'] + '\n\n' + state_block(r1[it['id']]))
            for it in ITEMS]
    p2 = WORK / f'preds_loop_{tag}_pass2.jsonl'
    judge_rows(agent_s, spec, p2, keeper1, f'{RUN1}:{tag}-pass2')
    r2 = read_rows(p2, 'P2ALL')
    sids = [it['id'] for it in ITEMS if it['id'] in r1 and it['id'] in r2]
    secondary[tag] = {'pass1': arm_scores(r1, sids), 'pass2_loop_all': arm_scores(r2, sids)}
    print(tag, secondary[tag])
    del agent_s
    torch.cuda.empty_cache()
print('secondary reads done (descriptive only, no verdict reads on them)')

In [ ]:
# ---- Run 1, steps 5+6: score the registered legs, write the results JSON ----
pass2_all = read_rows(preds2_path, 'P2ALL')
pass2_gate_rows = read_rows(preds2_path, 'P2GATE')
gated = {i: (pass2_gate_rows[i] if i in pass2_gate_rows else pass1[i]) for i in ids}

def leg_readout(rows, label):
    s = arm_scores(rows, ids)
    d = paired_resampling_delta(pass1, rows, ids)
    legs = {
        'leg1_uncertain_below_137': s['n_uncertain'] < 137,
        'leg2_headline_above_pass1_interval_above': (s['headline_acc'] > 0.0467
                                                     and d['ci95'][0] > 0),
        'leg3_incl_uncertain_at_or_above_0.5000': s['acc_incl_uncertain'] >= 0.5000,
        'leg4_falsified_at_or_above_0.0811': s['per_class_headline']['falsified'] >= 0.0811,
    }
    return {'arm': label, 'scores': s, 'delta_vs_pass1': d, 'legs': legs,
            'all_legs_pass': all(legs.values()),
            'strong_form_headline_at_or_above_0.4300': s['headline_acc'] >= 0.4300}

out_all = leg_readout(pass2_all, 'loop-all (primary)')
out_gate = leg_readout(gated, 'disagreement-gated (ablation)')
if out_all['all_legs_pass']:
    verdict = 'SUPPORTED'
elif out_gate['all_legs_pass']:
    verdict = 'GATED-ONLY'
else:
    verdict = 'NOT SUPPORTED'
results1 = {
    'run': RUN1, 'prd': 'swarm_tests/loop-test/PRD.md',
    'split': 'NewsCLIPpings validation, frozen 300 probe, pair-only; MMFakeBench test untouched',
    'pass1_reproduction': {'session': s_mine, 'stored': s_stored,
                           'n_verdict_deviations': len(deviations)},
    'primary': out_all, 'ablation': out_gate,
    'secondary_descriptive_only': secondary,
    'verdict': verdict,
}
rp = RESULTS / 'loop-test-results.json'
rp.write_text(json.dumps(results1, indent=1))
keeper1.save_verified(rp, 'loop-test-results.json')
print(json.dumps({'verdict': verdict, 'primary legs': out_all['legs'],
                  'gated legs': out_gate['legs']}, indent=1))
print('Results written to Drive:', rp)

In [ ]:
# ---- Run 1 RESUME cell ----
# Run this after re-running Setup and the keeper cell in a new session.
# It restores Run 1's verified progress from Drive and reports what is
# done. Then re-run Run 1's cells in order: finished (id, arm) pairs are
# skipped, so each cell continues where the dead session stopped.
k = Keeper('loop-test')
for name in ('preds_loop_pass1.jsonl', 'preds_loop_pass2.jsonl',
             'preds_loop_stageb_pass1.jsonl', 'preds_loop_stageb_pass2.jsonl',
             'preds_loop_stagea_pass1.jsonl', 'preds_loop_stagea_pass2.jsonl'):
    local = WORK / name
    if k.restore(name, local):
        rows = [json.loads(l) for l in open(local)]
        ok = [r for r in rows if r.get('status') == 'ok']
        from collections import Counter
        print(name, '->', dict(Counter(r['arm'] for r in ok)), 'ok rows')
print('Resume state restored. Re-run Run 1 cells in order.')

## Run 2: the sufficiency trigger race (N5)

What this run does: races three abstention triggers on the loop's
frozen probes, each scored as a failure predictor and as an abstention
trigger at matched coverage under abstention accounting. PRD:
`swarm_tests/sufficiency-race/PRD.md`. The triggers: (1) the
choice-screen disagreement signal, from Run 1's pass-1 read; (2) a
direct sufficiency probability from a third typed question, asked
once per item, wording fixed verbatim in the PRD; (3) a logistic probe
on the frozen checkpoint's backbone representations, fit only on the
Amendment 15 carve's 200 tune items and frozen before any probe item
is scored.

Dependency: this run consumes Run 1's outputs. Its first cell checks
for Run 1's results and pass-1 predictions and stops if they are not
there. The race verdict reads against pass-1 failure only.

What it reads: Run 1's results JSON and pass-1 predictions, the scale
probe checkpoint, the probe items and images, the 200 tune items and
their images and trainer records.

What it writes: sufficiency predictions, tune pass-1 predictions,
the fitted probe's scores, and a results JSON with the registered
race readouts, to Drive and printed below.

In [ ]:
# ---- Run 2, step 0: dependency gate on the loop test's outputs ----
RUN2 = 'sufficiency-race'
keeper2 = Keeper(RUN2)
loop_results_path = RESULTS / 'loop-test-results.json'
assert loop_results_path.exists(), \
    'DEPENDENCY GATE FAILED: Run 1 (loop test) has no results on Drive. Run it first.'
loop_results = json.loads(loop_results_path.read_text())
print('loop test verdict on record:', loop_results['verdict'])
loop_pass1_path = WORK / 'preds_loop_pass1.jsonl'
if not loop_pass1_path.exists():
    # Run 1's verified checkpoints live under its own slug folder.
    Keeper('loop-test').restore('preds_loop_pass1.jsonl', loop_pass1_path)
pass1 = read_rows(loop_pass1_path, 'P1')
assert len(pass1) == 300, f'loop pass-1 read covers {len(pass1)} items, not 300'
ids = [it['id'] for it in ITEMS if it['id'] in pass1]

# Failure labels, defined once in the PRD: pass-1 verdict not decided-correct.
def is_failure(row):
    lab, unc = A.verdict(row, A.THR_A)
    return not (not unc and lab == row['gold'])
FAIL = {i: is_failure(pass1[i]) for i in pass1}
print(f'pass-1 failures on the probe: {sum(FAIL.values())} of 300')

In [ ]:
# ---- Run 2, step 1: the sufficiency pass (wording fixed verbatim in the PRD) ----
SUFF_QUESTION = {"type": "noul",
                 "instructions": "Given only the image, its caption, and the information supplied with them, is the supplied information sufficient to decide whether the caption truthfully describes the image in its original context?",
                 "criteria": {"false": "The supplied information is not sufficient to decide whether the caption truthfully describes the image in its original context.",
                              "true": "The supplied information is sufficient to decide whether the caption truthfully describes the image in its original context."}}

def suff_prob(result):
    '''P(sufficient) from a predict() result for the single suff question.'''
    ans = result['answers']['suff']
    if 'probabilities' in ans and ans['probabilities']:
        return float(ans['probabilities']['true'])
    if 'noul' in ans:   # noul-type answers report P(true) in this field
        return float(ans['noul'])
    raise KeyError(f'cannot read sufficiency probability from answer: {ans}')

def sufficiency_pass(agent, items, images_dir, out_path, keeper, tag):
    out_path = Path(out_path)
    done = set()
    if out_path.exists():
        for line in open(out_path):
            r = json.loads(line)
            if r.get('status') == 'ok':
                done.add(r['id'])
    out = open(out_path, 'a')
    n_new = 0
    for it in items:
        if it['id'] in done:
            continue
        p = Path(images_dir) / it['image_path'] if 'image_path' in it else Path(images_dir) / it['image']
        ok, reason, img = vs.verify_image_bytes(p.read_bytes())
        if not ok:
            out.write(json.dumps({'id': it['id'], 'status': f'skip:{reason}'}) + '\n')
            out.flush()
            continue
        with torch.inference_mode():
            result = agent.predict({'context': it['caption'], 'image': img},
                                   {'suff': SUFF_QUESTION},
                                   n_permutations=VCFG['n_permutations'],
                                   batch_size=VCFG['batch_size'],
                                   prefix_cache=VCFG['prefix_cache'],
                                   temperature=VCFG['temperature'],
                                   strict=VCFG['strict'])
        out.write(json.dumps({'id': it['id'], 'gold': it['gold'],
                              'p_sufficient': suff_prob(result),
                              'status': 'ok'}) + '\n')
        out.flush()
        n_new += 1
        keeper.maybe_save([out_path], manifest={'run': tag, 'rows_done': len(done) + n_new})
    out.close()
    keeper.save_verified(out_path)
    print(f'{tag}: {n_new} new sufficiency rows')
    return out_path

TUNE_ITEMS = [json.loads(l) for l in open(INPUTS / 'scale_tune_items.jsonl')]
suff_probe_path = WORK / 'suff_probe.jsonl'
sufficiency_pass(agent_scale, ITEMS, PROBE_IMAGES, suff_probe_path, keeper2, RUN2 + ':probe')
TUNE_IMAGES = INPUTS / 'scale_tune_images'
suff_tune_path = WORK / 'suff_tune.jsonl'
sufficiency_pass(agent_scale, TUNE_ITEMS, TUNE_IMAGES, suff_tune_path, keeper2, RUN2 + ':tune')
print('sufficiency pass done (300 probe + 200 tune)')

In [ ]:
# ---- Run 2, steps 2+3: tune pass-1 labels, representations, the frozen probe ----
# Tune pass-1 read: trigger 3's failure labels come from these verdicts.
tune_items_path = WORK / 'tune_items_for_judge.jsonl'
with open(tune_items_path, 'w') as f:
    for it in TUNE_ITEMS:
        f.write(json.dumps(it) + '\n')
tune_preds_path = WORK / 'preds_tune_pass1.jsonl'
done = set()
if tune_preds_path.exists():
    for line in open(tune_preds_path):
        r = json.loads(line)
        if r.get('status') == 'ok':
            done.add((r['id'], r['arm']))
out = open(tune_preds_path, 'a')
for it in TUNE_ITEMS:
    if (it['id'], 'P1') in done:
        continue
    p = TUNE_IMAGES / it['image_path'] if 'image_path' in it else TUNE_IMAGES / it['image']
    ok, reason, img = vs.verify_image_bytes(p.read_bytes())
    if not ok:
        out.write(json.dumps({'id': it['id'], 'arm': 'P1', 'gold': it['gold'],
                              'status': f'skip:{reason}'}) + '\n')
        out.flush()
        continue
    with torch.inference_mode():
        result = agent_scale.predict({'context': it['caption'], 'image': img},
                                     {'native': V1_NATIVE, 'fake': V1_FAKE},
                                     n_permutations=VCFG['n_permutations'],
                                     batch_size=VCFG['batch_size'],
                                     prefix_cache=VCFG['prefix_cache'],
                                     temperature=VCFG['temperature'],
                                     strict=VCFG['strict'])
    dec = decode_answers(result, LABELS)
    out.write(json.dumps({'id': it['id'], 'arm': 'P1', 'gold': it['gold'],
                          'route': 'pair_only', 'status': 'ok', **dec}) + '\n')
    out.flush()
    keeper2.maybe_save([tune_preds_path], manifest={'run': RUN2 + ':tune-pass1'})
out.close()
keeper2.save_verified(tune_preds_path)
tune_rows = read_rows(tune_preds_path, 'P1')
print(f'tune pass-1 rows: {len(tune_rows)} of 200')
assert len(tune_rows) == 200, ('tune pass-1 read is incomplete; the representation '
                               'probe may not be fit on a partial or misaligned set')

In [ ]:
# ---- Run 2, step 3 continued: readout features and the logistic probe ----
# Features: the model forward's second output (the readout activations
# the trainer itself uses), mean-pooled per item, from the item's
# choice-form trainer example. The probe is fit on tune items ONLY and
# frozen before probe items are scored (PRD fix 2).
import numpy as np
import laya_trainer as lt

torch_mod, laya_mod, vlm_train = lt._laya_modules()
model = agent_scale.model
processor = agent_scale.processor

def readout_features(records_path, dataset='ncp'):
    examples = lt._examples(vlm_train, str(records_path), form=None, dataset=dataset)
    from laya.vlm import collate_vlm
    import functools
    stream = vlm_train.ItemStream(processor, examples, 0)
    loader = torch.utils.data.DataLoader(
        stream, batch_size=1, num_workers=0,
        collate_fn=functools.partial(collate_vlm, pad_id=processor.tokenizer.pad_token_id))
    feats, ids = [], []
    with torch.inference_mode():
        for batch in loader:
            b = {k: (v.to('cuda') if torch.is_tensor(v) else v) for k, v in batch.items()}
            logits, act = model(b['input_ids'], b['attention_mask'], b['marker_pos'],
                                b['marker_mask'], b['qtype'],
                                pixel_values=b.get('pixel_values'),
                                pixel_attention_mask=b.get('pixel_attention_mask'),
                                option_span=b.get('option_span'),
                                raw_pixels=b.get('raw_pixels'),
                                image_mask=b.get('image_mask'))
            a = act.float()
            feats.append(a.reshape(a.shape[0], -1).mean(dim=1).cpu().numpy())
    return np.concatenate(feats)

# NOTE: _examples returns the study's two-form records; the feature for an
# item is taken from its choice-form example, in file order, one row per
# example. The mapping example->item follows the records file's own order;
# the run log must print the counts so any mismatch is visible, never silent.
X_tune = readout_features(INPUTS / 'scale_tune_records.jsonl')
X_probe = readout_features(INPUTS / 'probe_records.jsonl')
print('feature shapes: tune', X_tune.shape, '| probe', X_probe.shape)
assert X_tune.shape[0] >= 200 and X_probe.shape[0] >= 300, \
    'feature rows fewer than the registered item sets; stopping rather than misaligning'

from sklearn.linear_model import LogisticRegression
tune_ids = [it['id'] for it in TUNE_ITEMS if it['id'] in tune_rows]
assert X_tune.shape[0] >= len(tune_ids), 'tune feature rows fewer than tune items'
y_tune = np.array([1.0 if is_failure(tune_rows[i]) else 0.0 for i in tune_ids])
probe_clf = LogisticRegression(C=1.0, max_iter=1000)
probe_clf.fit(X_tune[:len(tune_ids)], y_tune)
PROBE_SCORE = {}   # filled in the scoring cell from the probe-item features
np.save(WORK / 'X_probe.npy', X_probe)
keeper2.save_verified(WORK / 'X_probe.npy', 'X_probe.npy')
print('representation probe fit on tune items only and frozen:',
      'tune failures', int(y_tune.sum()), 'of', len(y_tune))

In [ ]:
# ---- Run 2, step 4: the race, scored exactly as registered ----
from sklearn.metrics import roc_auc_score

suff = {}
for line in open(suff_probe_path):
    r = json.loads(line)
    if r.get('status') == 'ok':
        suff[r['id']] = r
race_ids = [i for i in ids if i in suff]
assert len(race_ids) == 300, f'race covers {len(race_ids)} items, not 300'
y_fail = np.array([1.0 if FAIL[i] else 0.0 for i in race_ids])

score_disagreement = np.array([abs(pass1[i]['probabilities']['falsified']
                                   - pass1[i]['fake_probability']) for i in race_ids])
score_sufficiency = np.array([1.0 - suff[i]['p_sufficient'] for i in race_ids])
# Representation probe scores: features were extracted in probe-file order;
# if the counts disagree, stop rather than guess the alignment.
Xp = np.load(WORK / 'X_probe.npy')
assert Xp.shape[0] >= 300, 'probe feature rows fewer than probe items'
score_probe = probe_clf.predict_proba(Xp[:300])[:, 1]

def auc_interval(score_a, score_b, y, resamples=10000, seed=20261122):
    import random
    rng = random.Random(seed)
    n = len(y)
    diffs = []
    for _ in range(resamples):
        idx = [rng.randrange(n) for _ in range(n)]
        if len(set(y[idx])) < 2:
            continue
        diffs.append(roc_auc_score(y[idx], score_a[idx])
                     - roc_auc_score(y[idx], score_b[idx]))
    diffs.sort()
    m = len(diffs)
    return [round(diffs[int(0.025 * m)], 4), round(diffs[int(0.975 * m)], 4)]

aucs = {'disagreement': round(roc_auc_score(y_fail, score_disagreement), 4),
        'sufficiency': round(roc_auc_score(y_fail, score_sufficiency), 4),
        'representation_probe': round(roc_auc_score(y_fail, score_probe), 4)}
auc_diff_ci = auc_interval(score_sufficiency, score_disagreement, y_fail)

GRID = [round(0.05 * k, 2) for k in range(1, 21)]
def matched_coverage(score):
    order = np.argsort(-score)   # abstain on highest scores first
    curve = {}
    for cov in GRID:
        k = int(round(cov * len(race_ids)))
        decided_idx = order[:k] if cov < 1.0 else order
        decided = [race_ids[j] for j in decided_idx]
        s = arm_scores({i: pass1[i] for i in decided}, decided) if decided else None
        lost_correct = sum(1 for j in range(len(race_ids))
                           if j not in set(decided_idx.tolist())
                           and not FAIL[race_ids[j]])
        curve[str(cov)] = {'decided_acc': s['headline_acc'] if s else None,
                           'lost_correct': lost_correct,
                           'selective_risk': (round(1 - s['headline_acc'], 4) if s else None)}
    return curve

curves = {'disagreement': matched_coverage(score_disagreement),
          'sufficiency': matched_coverage(score_sufficiency),
          'representation_probe': matched_coverage(score_probe)}
band = [str(round(0.05 * k, 2)) for k in range(5, 16)]   # 0.25 to 0.75 decided
suff_band_ok = all(curves['sufficiency'][c]['decided_acc']
                   >= curves['disagreement'][c]['decided_acc'] for c in band)
if aucs['sufficiency'] > aucs['disagreement'] and auc_diff_ci[0] > 0 and suff_band_ok:
    race_verdict = 'SUFFICIENCY WINS'
elif (aucs['representation_probe'] > aucs['sufficiency']
      and aucs['representation_probe'] > aucs['disagreement']):
    race_verdict = 'PROBE WINS (claim becomes geometric, not typed)'
else:
    race_verdict = 'TIE OR SUFFICIENCY NOT SUPPORTED (disagreement stands)'
results2 = {'run': RUN2, 'prd': 'swarm_tests/sufficiency-race/PRD.md',
            'split': 'NewsCLIPpings validation, frozen 300 probe; probe fit on 200 tune items only; MMFakeBench test untouched',
            'aucs': aucs, 'sufficiency_minus_disagreement_auc_ci95': auc_diff_ci,
            'matched_coverage_curves': curves,
            'sufficiency_band_check_0.25_0.75': suff_band_ok,
            'verdict': race_verdict}
rp2 = RESULTS / 'sufficiency-race-results.json'
rp2.write_text(json.dumps(results2, indent=1))
keeper2.save_verified(rp2, 'sufficiency-race-results.json')
print(json.dumps({'aucs': aucs, 'auc diff CI': auc_diff_ci, 'verdict': race_verdict}, indent=1))
print('Results written to Drive:', rp2)

In [ ]:
# ---- Run 2 RESUME cell ----
# Re-run Setup and the keeper cell first. Restores Run 2's verified
# progress; then re-run Run 2's cells in order. Finished items are
# skipped by id in every pass.
k = Keeper('sufficiency-race')
for name in ('suff_probe.jsonl', 'suff_tune.jsonl', 'preds_tune_pass1.jsonl'):
    local = WORK / name
    if k.restore(name, local):
        rows = [json.loads(l) for l in open(local)]
        print(name, '->', sum(1 for r in rows if r.get('status') == 'ok'), 'ok rows')
print('Resume state restored. Re-run Run 2 cells in order.')

## Run 3: the W1 retrain

What this run does: retrains the lost W1 checkpoint on the exact
recorded recipe (245 steps, soft_ce, full arm, vision frozen, seed 0,
lr_encoder 1e-5, lr_head 1e-3, batch 8, the study's own
`train_mmf.py` driver, unmodified) and delivers a complete, verified
checkpoint to persistent storage. PRD: `swarm_tests/w1-retrain/PRD.md`.
Success unblocks the loss-masking audit's step 0 gate; this run
produces no audit number and no claim.

What it reads: the laya-vision-base checkpoint from the inputs bundle
(identity-gated against the recorded config and the exact trainable
parameter count, 114,728,323), `mmf_train_base.jsonl` (980 records),
and `mmf_tune_base.jsonl`.

What it writes: the retrained checkpoint folder, verified locally and
again on Drive (safetensors header completeness, expected 804,696,880
bytes and 416 tensors, plus a clean load), and a results JSON with the
run's fingerprints beside the original's, including the registered
tune smoke band (macro-F1 within 0.05 of 0.6533, else the checkpoint
is labelled SUSPECT).

In [ ]:
# ---- Run 3, step 0: base identity gate (fail-closed) ----
import subprocess, json as _json
RUN3 = 'w1-retrain'
keeper3 = Keeper(RUN3)
BASE_DIR = INPUTS / 'ckpts' / 'laya-vision-base'
cfg = _json.loads((BASE_DIR / 'vlm_agent_config.json').read_text())
expected_cfg = {'backbone': 'HuggingFaceTB/SmolVLM-256M-Instruct', 'head_layers': 2,
                'n_act': 2, 'max_len': 1024, 'head_max_len': 256,
                'option_attention': 'block', 'image_size': 512, 'readout': 'terminator'}
for k_, v_ in expected_cfg.items():
    assert cfg.get(k_) == v_, f'BASE IDENTITY GATE FAILED: config {k_} is {cfg.get(k_)}, expected {v_}'
n_train = sum(1 for _ in open(INPUTS / 'mmf_train_base.jsonl'))
assert n_train == 980, f'BASE IDENTITY GATE FAILED: train file has {n_train} records, expected 980'

torch_mod, laya_mod, vlm_train = lt._laya_modules()
_agent = laya_mod.load_vlm(str(BASE_DIR))
vlm_train.set_trainable(_agent.model, 'full')
n_trainable = lt._trainable_count(_agent.model)
print('trainable parameters:', n_trainable)
assert n_trainable == 114728323, \
    f'BASE IDENTITY GATE FAILED: trainable count {n_trainable} != 114,728,323'
del _agent
torch.cuda.empty_cache()
print('BASE IDENTITY GATE PASSED')

In [ ]:
# ---- Run 3, steps 1+2: timing smoke, then the exact recorded training run ----
import subprocess, time as _time
OUT3 = WORK / 'w1_retrain_out'
SMOKE = WORK / 'w1_smoke_out'
def run_train(steps, out_dir):
    cmd = ['python3', str(WORK / 'train_mmf.py'),
           '--base', str(BASE_DIR),
           '--train-file', str(INPUTS / 'mmf_train_base.jsonl'),
           '--tune-file', str(INPUTS / 'mmf_tune_base.jsonl'),
           '--out-dir', str(out_dir), '--steps', str(steps)]
    t0 = _time.monotonic()
    r = subprocess.run(cmd, capture_output=True, text=True)
    dt = _time.monotonic() - t0
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError('train_mmf.py failed')
    return dt

# The bundle's records resolve their images relative to the inputs
# folder layout the study used; train_mmf.py runs from the inputs dir.
import os
os.chdir(str(INPUTS))
dt30 = run_train(30, SMOKE)
projected = dt30 / 30 * 245
print(f'30-step smoke: {dt30:.1f}s | projected 245 steps: {projected:.1f}s')
assert projected <= 30 * 60, ('TIMING GATE: projected run exceeds 30 minutes; '
                              'stopping rather than training uncheckpointed (PRD step 1)')
dt = run_train(245, OUT3)
print(f'training done in {dt:.1f}s')
result3 = _json.loads((OUT3 / 'result.json').read_text())
print({k: result3[k] for k in ('steps', 'seed', 'loss_first', 'loss_last',
                               'tune_macro_f1', 'tune_accuracy') if k in result3})

In [ ]:
# ---- Run 3, steps 3+4: verify, pull to Drive verified, smoke band, results ----
model3 = OUT3 / 'model.safetensors'
chk3 = safetensors_header_check(model3)
print('retained checkpoint header check:', chk3)
assert chk3['ok'] and chk3['bytes'] == 804696880 and chk3['tensors'] == 416, \
    'VERIFY FAILED: retrained checkpoint is not header-complete at the recorded size'
_agent3 = laya_mod.load_vlm(str(OUT3))
print('clean load through laya.load_vlm: OK')
del _agent3
torch.cuda.empty_cache()

dest3 = keeper3.dir / 'w1_checkpoint'
dest3.mkdir(parents=True, exist_ok=True)
for f in sorted(OUT3.rglob('*')):
    if f.is_file():
        keeper3.save_verified(f, 'w1_checkpoint/' + str(f.relative_to(OUT3)))
print('checkpoint copied to Drive with every file verified on save')

tune_f1 = result3.get('tune_macro_f1')
band_ok = tune_f1 is not None and abs(tune_f1 - 0.6533) <= 0.05
results3 = {'run': RUN3, 'prd': 'swarm_tests/w1-retrain/PRD.md',
            'recipe': 'train_mmf.py unmodified; full arm; soft_ce; vision frozen; '
                      '245 steps; seed 0; lr_encoder 1e-5; lr_head 1e-3; batch 8',
            'fingerprints_new': {'loss_first': result3.get('loss_first'),
                                 'loss_last': result3.get('loss_last'),
                                 'tune_macro_f1': tune_f1,
                                 'tune_accuracy': result3.get('tune_accuracy'),
                                 'trainable_params': result3.get('trainable_params')},
            'fingerprints_original': {'loss_first': 1.7355, 'loss_last': 0.0966,
                                      'tune_macro_f1': 0.6533, 'tune_accuracy': 0.6571,
                                      'trainable_params': 114728323},
            'header_check': chk3, 'tune_smoke_band_within_0.05': band_ok,
            'label': 'COMPLETE' if band_ok else 'SUSPECT (outside tune smoke band; board review before the audit proceeds)',
            'drive_location': str(dest3)}
rp3 = RESULTS / 'w1-retrain-results.json'
rp3.write_text(json.dumps(results3, indent=1))
keeper3.save_verified(rp3, 'w1-retrain-results.json')
print(json.dumps(results3, indent=1))

In [ ]:
# ---- Run 3 RESUME cell ----
# Training is a single monolithic 245-step call (minutes on a T4), so
# there is no mid-training position to restore: the registered resume
# is (a) if a verified complete checkpoint already sits on Drive, do
# not retrain; (b) otherwise re-run Run 3 from step 0. The recipe is
# deterministic (seed 0, seeded item stream), so a restart reproduces
# the run rather than resuming a partial one. A partial checkpoint is
# never labelled complete.
dest = CKPT_ROOT / 'w1-retrain' / 'w1_checkpoint' / 'model.safetensors'
if dest.exists():
    chk = safetensors_header_check(dest)
    print('existing Drive checkpoint:', chk)
    print('COMPLETE and verified: do not retrain.' if chk['ok'] and chk['bytes'] == 804696880
          else 'Present but not verified complete: re-run Run 3 from step 0.')
else:
    print('no Drive checkpoint yet: re-run Run 3 cells from step 0.')

## After the runs

The three results files on Drive (`results/loop-test-results.json`,
`results/sufficiency-race-results.json`, `results/w1-retrain-results.json`)
and the W1 checkpoint folder are the deliverables. Back on the study
VM they are recorded into research-brain with read-back, and the
loss-masking audit's step 0 gate is re-run against the new W1
checkpoint under its own PRD.

Do not re-run a scoring cell to see whether a verdict changes. Each
run's verdict is read once, off the registered legs, from the full
frozen item set. If a number looks wrong, stop and report it; the
record is corrected by registration, never by re-rolling.

The MMFakeBench test set remains closed. Nothing in this notebook
touched it.